In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from scipy.spatial import distance
from scipy import spatial
import numpy as np

In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [3]:
topics= {
"tech":"Technology",
"science":"Science",
"sport":"Sports",
"business":"Business"
}

topic_labels=[topic for topic in topics.values()]

In [4]:
def create_embeddings(texts):
    response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
    )
    response_dict = response.model_dump()
    
    return [data['embedding'] for data in response_dict['data']]

topic_embeddings=create_embeddings(topic_labels)


In [5]:
article= {
"headline":"Tech company launches new AI chip",
"keywords": ["AI","hardware","technology"]
}

In [6]:
def create_article_text(article):
    return f"""
    Headline:{article['headline']}
    Keywords:{', '.join(article['keywords'])}
    """

In [7]:
article_text=create_article_text(article)

article_embedding=create_embeddings([article_text])[0]


In [8]:
def find_closest_label(query_vector,embeddings):

   distances= []

   for i,embedding in enumerate(embeddings):
      distance=spatial.distance.cosine(query_vector,embedding)
      distances.append({
"index":i,
"distance":distance
        })
      closest=min(distances, key=lambda x:x["distance"])

   return closest

In [9]:
result=find_closest_label(article_embedding, topic_embeddings)

print(result)

{'index': 0, 'distance': np.float64(0.6894567735469135)}


In [10]:
predicted_label=topic_labels[result["index"]]

print(predicted_label)

Technology


In [11]:
topics= [{
"label":"Technology", "description":"Technology news related to computers, AI, software, and hardware" },
{"label":"Science", "description":"Scientific discoveries, research, and scientific studies"},
{"label":"Sports", "description":"Sports events, competitions, teams, and athletes"},
{"label":"Business", "description":"Business news, companies, finance, and economic markets"}
]


In [12]:
def convert_to_text(text):
    return f"""
    Label: {text['label']}
    Description: {text['description']}

"""

In [13]:
topic_text = [convert_to_text(topic) for topic in topics]

topic2_embeddings = create_embeddings(topic_text)

result=find_closest_label(article_embedding, topic2_embeddings)

In [14]:
predicted_label=topics[result["index"]]['label']

print(predicted_label)

Technology
